# ML-04 — Search Intelligence Data Contract

This notebook defines my content-refresh ranking slice, verifies the warehouse grain and February feature window, builds five leakage-safe features, and demonstrates the label-leakage trap.

> Prediction point: **end of February 2026**. March is the outcome window, so no March information is used as a feature.

## 1. Contract in plain words

**1. One row means:** one pseudonymized content item for one pseudonymized client (`client_hash_id × content_hash_id`), after aggregating the daily fact to the February decision window.

**2. Tables:** `fact_content_daily_performance` for February search-performance features and March outcomes; `dim_content` only for stable content metadata and the publication/creation filters.

**3. Time window:** February 1–28, 2026 is the feature/decision window; March 1–31, 2026 is the forward outcome window. The windows do not overlap.

**4. What I predict/rank:** `went_dark = 1` when a page with enough February search activity records zero GSC clicks during March; the model output is a **refresh-priority risk ranking**, not proof that a refresh will cause recovery.

**5. Deliberately excluded:** future optimization/update fields and March performance fields are excluded because they were not knowable at the February decision point. I also exclude the fixed 90-day query table because its window overlaps the outcome period.

In [ ]:
# Setup: remote Parquet through DuckDB. The HF token is never written into SQL text.
import os
import duckdb
import pandas as pd
import numpy as np

try:
    from google.colab import userdata
except ImportError:
    userdata = None

hf_token = os.environ.get("HF_TOKEN")
if not hf_token and userdata is not None:
    hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise RuntimeError("Set HF_TOKEN as a Colab Secret or environment variable before running this notebook.")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute("SET VARIABLE hf_token = ?", [hf_token])
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN getvariable('hf_token'))")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
DIM = f"{REL}/dim_content.parquet"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected. Feature window: February 2026. Outcome window: March 2026.")


## 2. Verify three facts

### Query 1 — Grain
The raw fact should be one row per `report_date × client_hash_id × content_hash_id`; duplicate groups should be zero.

In [ ]:
# Verification query 1 — grain
grain_check = con.sql(f"""
SELECT COUNT(*) AS duplicate_groups
FROM (
    SELECT report_date, client_hash_id, content_hash_id
    FROM {FEB}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
)
""").df()

grain_check

### Query 2 — Row count and date span
This checks that the mid-panel feature partition is the February 2026 slice and records its actual size.

In [ ]:
# Verification query 2 — row count and date span for the February slice
feb_span = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(DISTINCT report_date) AS distinct_dates
FROM {FEB}
""").df()

feb_span

### Query 3 — Availability
The GSC filter deliberately uses `IS TRUE`. Rows where availability is FALSE or NULL are not treated as measured search data.

In [ ]:
# Verification query 3 — availability
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
FROM {FEB}
""").df()

availability_check

## 3. Five features + the deliberate leakage trap

The feature frame is one row per `client_hash_id × content_hash_id`. I keep **five features maximum**:

1. **`feb_impressions`** — knowable at the decision moment because it is the total GSC exposure observed through February 28.
2. **`feb_clicks`** — knowable at the decision moment because it is the total GSC click history observed through February 28.
3. **`feb_ctr`** — knowable at the decision moment because it is computed only from February clicks and impressions.
4. **`feb_position`** — knowable at the decision moment because it is the February impression-weighted average GSC position.
5. **`content_age_days`** — knowable at the decision moment because it is derived from the content creation date and the February 28 cutoff; it is only kept for content that existed by the cutoff.

The ranking target is `went_dark`, defined from **March clicks**. March fields are labels/outcomes, never features.

In [ ]:
# Build the February feature frame and the March label.
# The query reads only small aggregated results into pandas.
feature_sql = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS feb_impressions,
        SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS feb_clicks,
        SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE)
            / NULLIF(SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE), 0) AS feb_position
    FROM {FEB}
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) >= 100
       AND SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) >= 3
),
march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS march_clicks,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS march_measured_days
    FROM {MAR}
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.feb_impressions,
    f.feb_clicks,
    f.feb_clicks / NULLIF(f.feb_impressions, 0) AS feb_ctr,
    f.feb_position,
    DATE_DIFF('day', d.content_created_date, DATE '2026-02-28') AS content_age_days,
    COALESCE(m.march_clicks, 0) AS march_clicks,
    COALESCE(m.march_measured_days, 0) AS march_measured_days
FROM feb f
JOIN {DIM} d USING (client_hash_id, content_hash_id)
LEFT JOIN march m USING (client_hash_id, content_hash_id)
WHERE d.is_published IS TRUE
  AND d.content_created_date <= DATE '2026-02-28'
""").df()

# A label is defined only when March has at least one measured GSC day.
feature_sql = feature_sql[feature_sql["march_measured_days"] > 0].copy()
feature_sql["went_dark"] = (feature_sql["march_clicks"] == 0).astype(int)

feature_frame = feature_sql[
    ["client_hash_id", "content_hash_id",
     "feb_impressions", "feb_clicks", "feb_ctr",
     "feb_position", "content_age_days", "went_dark"]
].copy()

print(f"Feature rows: {len(feature_frame):,}")
print(f"March-labelled rows: {len(feature_sql):,}")
print(f"went_dark positives: {feature_frame['went_dark'].sum():,}")
print(f"went_dark rate: {feature_frame['went_dark'].mean():.1%}")
feature_frame.head(10)

### The trap: deliberately add one label-derived feature

I now add `leak_from_label`, which is derived directly from `went_dark`. This is intentionally dishonest: it contains the answer at prediction time. A quick decision-tree score should therefore become essentially perfect.

Then I delete it and keep the honest score from the five February features. The point is not the model itself; it is demonstrating why a feature that is mathematically derived from the label can create a meaningless near-perfect score.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

feature_cols = [
    "feb_impressions",
    "feb_clicks",
    "feb_ctr",
    "feb_position",
    "content_age_days",
]

model_df = feature_frame[feature_cols + ["went_dark"]].dropna().copy()

X_train, X_test, y_train, y_test = train_test_split(
    model_df[feature_cols],
    model_df["went_dark"],
    test_size=0.30,
    random_state=42,
    stratify=model_df["went_dark"],
)

honest_tree = DecisionTreeClassifier(max_depth=4, random_state=42)
honest_tree.fit(X_train, y_train)
honest_score = accuracy_score(y_test, honest_tree.predict(X_test))

# Deliberate leak: the feature is the label itself.
leaky = model_df.copy()
leaky["leak_from_label"] = leaky["went_dark"]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    leaky[feature_cols + ["leak_from_label"]],
    leaky["went_dark"],
    test_size=0.30,
    random_state=42,
    stratify=leaky["went_dark"],
)

leaky_tree = DecisionTreeClassifier(max_depth=4, random_state=42)
leaky_tree.fit(X_train_l, y_train_l)
leaky_score = accuracy_score(y_test_l, leaky_tree.predict(X_test_l))

print(f"Honest accuracy (5 features): {honest_score:.3f}")
print(f"Leaky accuracy (+ label-derived column): {leaky_score:.3f}")
print(f"Leakage jump: {leaky_score - honest_score:+.3f}")

# Remove the trap before keeping the feature frame.
feature_frame = feature_frame.drop(columns=["went_dark"])
print("Final feature frame columns:", feature_frame.columns.tolist())

## 4. Named limitation

**Limitation: the panel is unbalanced and client-concentrated.** Client histories do not all cover the full feature window, so a February aggregate can represent fewer observed days for some clients. The warehouse also has an unbalanced client panel, so a random row split can make performance look better than it generalises to a new client. The model should therefore use client-grouped validation later, and feature totals should be interpreted alongside observed-data coverage.

A second limitation is that `dim_content` is a current snapshot rather than a full historical dimension, so content metadata can be less certain as-of February. For this first five-feature frame I avoid snapshot-sensitive fields and keep only content age.

## 5. Self-check

- [x] Five plain-words contract answers are stated.
- [x] Exactly three verification queries are clearly marked; availability uses `IS TRUE`.
- [x] The feature frame contains five decision-time features.
- [x] Every feature has an “knowable at the decision moment because…” line.
- [x] One label-derived leakage column is deliberately added, measured, then removed.
- [x] One named limitation is documented.
- [ ] Notebook executed end-to-end with the gated warehouse and committed with outputs.
- [x] No HF token is stored in the notebook.
